# NOVA 01b — MOD-01 Obstacle Detection v1.2.0 (COCO baseline)
**No training.** Takes stock COCO-pretrained YOLOv8n (80 real class names), quantizes to INT8 @ 320px, evaluates on COCO128, benchmarks, and publishes to `unixio/nova-obstacle-detection` as v1.2.0.

Rationale: Detectra/VisDrone fine-tunes (v1.0.0/v1.1.0) reached only ~10% mAP50 — kept on the Hub as ablations. The stock model satisfies SRS FR-01-03 ('80-class COCO dataset') with real TTS-speakable names.

**~10 minutes total. No datasets to attach.** GPU + Internet + HF_TOKEN.

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
# Pre-install the LiteRT converter stack so Ultralytics' AutoUpdate doesn't
# swap numpy's deps mid-kernel (that corrupts the running Python process).
!pip install -q ultralytics 'litert-torch>=0.9.0' 'ai-edge-litert>=2.1.4' ai-edge-quantizer

In [ ]:
# Evaluate stock YOLOv8n on COCO128 at 320px (deployment resolution).
# COCO128 is small — cite Ultralytics' official full-COCO numbers alongside
# (YOLOv8n: 37.3 mAP50-95 @ 640) in the report.
import json, os
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
metrics = model.val(data='coco128.yaml', imgsz=320)
results = {
    'mAP50_coco128@320': float(metrics.box.map50),
    'mAP50-95_coco128@320': float(metrics.box.map),
    'official_coco_mAP50-95@640': 0.373,
    'note': 'stock COCO-pretrained YOLOv8n, no fine-tune',
}
os.makedirs('/kaggle/working/evaluation', exist_ok=True)
json.dump(results, open('/kaggle/working/evaluation/obstacle_results.json', 'w'), indent=2)
print(json.dumps(results, indent=2))

In [ ]:
# Write the 80 COCO class names — the app's TTS reads these
os.makedirs('/kaggle/working/labels', exist_ok=True)
names = [model.names[i] for i in range(len(model.names))]
open('/kaggle/working/labels/coco_labels.txt', 'w').write('\n'.join(names) + '\n')
print(len(names), 'classes:', names[:8], '...')

In [ ]:
# INT8 quantization at 320 (COCO128 calibration), run in a FRESH subprocess
# via the yolo CLI — immune to any in-kernel package-state issues.
!yolo export model=yolov8n.pt format=litert quantize=int8 imgsz=320 data=coco128.yaml
import glob
candidates = glob.glob('**/yolov8n*int8*.tflite', recursive=True) + \
             glob.glob('**/yolov8n*_int8.tflite', recursive=True)
print('candidates:', candidates)
assert candidates, 'INT8 TFLite not found after export'
exported = candidates[0]
size_mb = os.path.getsize(exported) / 1e6
print(f'{exported}: {size_mb:.2f} MB (budget: <15 MB)')
assert size_mb < 15

In [ ]:
# Publish v1.2.0
!python scripts/push_to_huggingface.py --module MOD-01 \
    --tflite {exported} \
    --labels /kaggle/working/labels/coco_labels.txt \
    --eval-json /kaggle/working/evaluation/obstacle_results.json --version 1.2.0